# Expectation Values

MatchCake can efficiently compute expectation values of Hamiltonians of the form

$$
\mathcal{H} = \sum_{k=0}^{K-1} \alpha_k P_k,
$$

where each $P_k$ is an arbitrary Pauli word, for any matchgate circuit $V$ applied to
any qubit product state $|\psi_\text{prod}\rangle = \bigotimes_{k=0}^{n-1}(a_k|0\rangle + b_k|1\rangle)$:

$$
\langle\mathcal{H}\rangle = \langle\psi_\text{prod}|V^\dagger \mathcal{H} V|\psi_\text{prod}\rangle.
$$

This includes Pauli words that break total fermionic parity (e.g. $Z_k I_{k+1}$,
$I_k Z_{k+1}$) as well as superposition initial states such as $|+\rangle^{\otimes n}$.

The algorithm is based on the extended covariance matrix formalism. See
[`docs/expectation_values_theory.md`](../docs/expectation_values_theory.md) for the
full derivation and references.

In [1]:
import numpy as np
import pennylane as qml

import matchcake as mc
from matchcake.operations.state_preparation import ProductState

## Hamiltonian

We construct a Hamiltonian on 3 qubits that mixes parity-preserving and
parity-breaking Pauli words.

In [2]:
hamiltonian = qml.Hamiltonian(
    [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    [
        qml.X(0) @ qml.X(1),
        qml.Y(1) @ qml.Y(2),
        qml.Z(0) @ qml.Z(1),
        qml.Y(1) @ qml.X(2),
        qml.X(0) @ qml.Y(1),
        qml.Z(1) @ qml.I(2),
        qml.I(1) @ qml.Z(2),
    ],
)
n = len(hamiltonian.wires)
sptm = mc.operations.SingleParticleTransitionMatrixOperation.random(wires=hamiltonian.wires, seed=0)
U = sptm.to_qubit_operation()
print(hamiltonian)

0.1 * (X(0) @ X(1)) + 0.2 * (Y(1) @ Y(2)) + 0.3 * (Z(0) @ Z(1)) + 0.4 * (Y(1) @ X(2)) + 0.5 * (X(0) @ Y(1)) + 0.6 * (Z(1) @ I(2)) + 0.7 * (I(1) @ Z(2))


## Basis state initialization

We initialize the system in a computational-basis product state using
`ProductState.from_basis_state` and verify against `default.qubit`.

In [3]:
state_prep = ProductState.from_basis_state([1, 0, 1], wires=hamiltonian.wires)


@qml.qnode(mc.NIFDevice(n))
def circuit():
    state_prep.queue()
    sptm.queue()
    return qml.expval(hamiltonian)


@qml.qnode(qml.device("default.qubit", wires=n))
def svs_circuit():
    state_prep.queue()
    U.queue()
    return qml.expval(hamiltonian)


expval = circuit()
svs_expval = svs_circuit()
print(f"{expval = :.6f}")
print(f"{svs_expval = :.6f}")
np.testing.assert_allclose(expval, svs_expval, atol=1e-5)
print("Results agree with state-vector simulation.")

expval = -0.787843
svs_expval = -0.787843
Results agree with state-vector simulation.


## Arbitrary product state initialization

`ProductState` accepts arbitrary per-qubit amplitudes $(a_k, b_k)$, enabling
superposition initial states. Here we initialize all qubits in $|+\rangle =
\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and evaluate the same Hamiltonian,
including the parity-breaking terms $Z_1 I_2$ and $I_1 Z_2$.

In [4]:
plus_amplitudes = np.full((n, 2), 1.0 / np.sqrt(2), dtype=complex)
state_prep_plus = ProductState(plus_amplitudes, wires=hamiltonian.wires)


@qml.qnode(mc.NIFDevice(n))
def circuit_plus():
    state_prep_plus.queue()
    sptm.queue()
    return qml.expval(hamiltonian)


@qml.qnode(qml.device("default.qubit", wires=n))
def svs_circuit_plus():
    state_prep_plus.queue()
    U.queue()
    return qml.expval(hamiltonian)


expval_plus = circuit_plus()
svs_expval_plus = svs_circuit_plus()
print(f"{expval_plus = :.6f}")
print(f"{svs_expval_plus = :.6f}")
np.testing.assert_allclose(expval_plus, svs_expval_plus, atol=1e-5)
print("Results agree with state-vector simulation.")

expval_plus = 0.066344
svs_expval_plus = 0.066344
Results agree with state-vector simulation.


## Mixed product state

Any per-qubit amplitudes are valid. Below we mix $|+\rangle$, $|0\rangle$, and
a $y$-rotated state to demonstrate the general case.

In [5]:
theta = np.pi / 3
mixed_amplitudes = np.array(
    [
        [1.0, 1.0],
        [1.0, 0.0],
        [np.cos(theta / 2), np.sin(theta / 2)],
    ],
    dtype=complex,
)
mixed_amplitudes /= np.linalg.norm(mixed_amplitudes, axis=1, keepdims=True)
state_prep_mixed = ProductState(mixed_amplitudes, wires=hamiltonian.wires)


@qml.qnode(mc.NIFDevice(n))
def circuit_mixed():
    state_prep_mixed.queue()
    sptm.queue()
    return qml.expval(hamiltonian)


@qml.qnode(qml.device("default.qubit", wires=n))
def svs_circuit_mixed():
    state_prep_mixed.queue()
    U.queue()
    return qml.expval(hamiltonian)


expval_mixed = circuit_mixed()
svs_expval_mixed = svs_circuit_mixed()
print(f"{expval_mixed = :.6f}")
print(f"{svs_expval_mixed = :.6f}")
np.testing.assert_allclose(expval_mixed, svs_expval_mixed, atol=1e-5)
print("Results agree with state-vector simulation.")

expval_mixed = -0.169835
svs_expval_mixed = -0.169835
Results agree with state-vector simulation.


## Further reading

For the mathematical details behind this computation, the extended Majorana algebra,
the displacement vector, the $(2n+1)\times(2n+1)$ covariance matrix, the SPTM lift,
and the Pfaffian formula, see
[`docs/expectation_values_theory.md`](../docs/expectation_values_theory.md).